# 02 — Pré-processamento

**Dataset:** Vehicle Collision Data in Seattle (2005–2019)

**Tarefa:** Tratamento estrutural e padronização dos dados, com base nas conclusões da EDA (Bloco 1).

## Estrutura deste notebook

1. Carregamento
2. Extrair features temporais de DATE/TIME (mês, dia da semana, hora)
3. Remover colunas irrelevantes (`Unnamed: 0`, `SPDCASENO`, `DATE`, `TIME`)
4. Remover leakage (`INJURIES`, `SERIOUSINJURIES`, `FATALITIES`)
5. Remover multicolinearidade (`TMAX`, `TMIN`, `WSF5`)
6. Remover colunas com > 50% nulos (`response_type`, `response_time`)
7. Remover linhas duplicadas
8. Imputar `SNOW` e `SNWD` com 0
9. Imputar demais nulos (mediana p/ numéricas, moda p/ categóricas)
10. Tratar outliers (apenas `PERSONCOUNT` e `VEHCOUNT`)
11. Converter booleanas para int (0/1)
12. One-Hot Encoding das categóricas nominais
13. Manter `LIGHTCOND` como ordinal inteiro (já está nesse formato)
14. Split 80/20 estratificado por `SEVERITYCODE` (antes do scaler)
15. StandardScaler nas features contínuas (fit só no treino)
16. Verificar shapes finais
17. Salvar conjuntos de treino/teste

## 1. Carregamento

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

caminho = 'data/seattle_collision_data_2005_2019.csv'
if not os.path.exists(caminho):
    caminho = '/content/seattle_collision_data_2005_2019.csv'

df = pd.read_csv(caminho)
print("Shape inicial do dataset:", df.shape)

Shape inicial do dataset: (111882, 34)


## 2. Extrair features temporais de DATE/TIME

`DATE` está no formato `YYYY-MM-DD` → extraímos mês e dia da semana.

`TIME` está em **horas decimais** (ex.: `14.53` ≈ 14h32, intervalo de 0 a 23.98), e não em formato HHMM. Para obter a hora do dia basta truncar para inteiro.

In [2]:
df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')

df['mes'] = df['DATE'].dt.month
df['dia_da_semana'] = df['DATE'].dt.dayofweek
df['hora'] = df['TIME'].fillna(df['TIME'].median()).astype(int)

print("Features temporais extraídas.")
print("Intervalo de hora:", df['hora'].min(), "-", df['hora'].max())
display(df[['mes', 'dia_da_semana', 'hora']].head())

Features temporais extraídas.
Intervalo de hora: 0 - 23


,mes,dia_da_semana,hora
0,1,0,2
1,1,0,7
2,1,0,7
3,1,0,10
4,1,0,10


## 3. Remover colunas irrelevantes: Unnamed: 0, SPDCASENO, DATE, TIME

In [3]:
df = df.drop(columns=['Unnamed: 0', 'SPDCASENO', 'DATE', 'TIME'], errors='ignore')

print("Colunas irrelevantes removidas.")
print("Shape atual:", df.shape)

Colunas irrelevantes removidas.
Shape atual: (111882, 33)


## 4. Remover leakage: INJURIES, SERIOUSINJURIES, FATALITIES

In [4]:
df = df.drop(columns=['INJURIES', 'SERIOUSINJURIES', 'FATALITIES'], errors='ignore')

print("Colunas de leakage removidas.")
print("Shape atual:", df.shape)

Colunas de leakage removidas.
Shape atual: (111882, 30)


## 5. Remover multicolinearidade: TMAX, TMIN, WSF5

In [5]:
df = df.drop(columns=['TMAX', 'TMIN', 'WSF5'], errors='ignore')

print("Colunas multicolineares removidas.")
print("Shape atual:", df.shape)

Colunas multicolineares removidas.
Shape atual: (111882, 27)


## 6. Remover colunas com > 50% nulos: response_type, response_time

In [6]:
df = df.drop(columns=['response_type', 'response_time'], errors='ignore')

print("Colunas com mais de 50% de nulos removidas.")
print("Shape atual:", df.shape)

Colunas com mais de 50% de nulos removidas.
Shape atual: (111882, 25)


## 7. Remover linhas duplicadas

Após remover as colunas de ID e as não utilizadas, removemos linhas integralmente duplicadas (registros idênticos em todas as colunas restantes).

In [7]:
linhas_antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
linhas_depois = len(df)

print(f"Linhas duplicadas removidas: {linhas_antes - linhas_depois}")
print("Shape atual:", df.shape)

Linhas duplicadas removidas: 33
Shape atual: (111849, 25)


## 8. Imputar SNOW e SNWD com 0

In [8]:
df['SNOW'] = df['SNOW'].fillna(0)
df['SNWD'] = df['SNWD'].fillna(0)

print("Valores nulos de SNOW e SNWD substituídos por 0.")
print("Nulos em SNOW:", int(df['SNOW'].isnull().sum()), "| Nulos em SNWD:", int(df['SNWD'].isnull().sum()))

Valores nulos de SNOW e SNWD substituídos por 0.
Nulos em SNOW: 0 | Nulos em SNWD: 0


## 9. Imputar demais nulos com mediana (numéricas) ou moda (categóricas)

In [9]:
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

cat_cols = df.select_dtypes(include=['str', 'object', 'category']).columns
for col in cat_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Imputação finalizada.")
print("Total de valores nulos no dataset:", int(df.isnull().sum().sum()))

Imputação finalizada.
Total de valores nulos no dataset: 0


## 10. Tratar outliers

Apenas `PERSONCOUNT` (varia até 93) e `VEHCOUNT` (até 15) têm cauda longa que justifica capping no percentil 99.

`PEDCOUNT` (0–6) e `PEDCYLCOUNT` (0–2) têm intervalo natural pequeno: seus valores altos são raros, porém **legítimos e preditivos** (acidentes com pedestres/ciclistas tendem a ser mais graves). Capá-los destruiria sinal, por isso são mantidos.

In [10]:
cols_outliers = ['PERSONCOUNT', 'VEHCOUNT']

for col in cols_outliers:
    limite_superior = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=limite_superior)
    print(f"'{col}': valores limitados ao teto de {limite_superior}")

display(df[['PERSONCOUNT', 'PEDCOUNT', 'PEDCYLCOUNT', 'VEHCOUNT']].describe().loc[['max']])

'PERSONCOUNT': valores limitados ao teto de 7.0
'VEHCOUNT': valores limitados ao teto de 4.0


,PERSONCOUNT,PEDCOUNT,PEDCYLCOUNT,VEHCOUNT
max,7.0,6.0,2.0,4.0


## 11. Converter booleanas para int (0/1)

In [11]:
bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].astype(int)

print("Booleanas convertidas para inteiros (0/1).")
print("Colunas convertidas:", list(bool_cols))

Booleanas convertidas para inteiros (0/1).
Colunas convertidas: ['INATTENTIONIND', 'UNDERINFL', 'SPEEDING', 'HITPARKEDCAR', 'intersection_related']


## 12. One-Hot Encoding: COLLISIONTYPE, WEATHER, ROADCOND, JUNCTIONTYPE

In [12]:
cols_to_encode = ['COLLISIONTYPE', 'WEATHER', 'ROADCOND', 'JUNCTIONTYPE']
df = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)

ohe_cols = df.select_dtypes(include=['bool']).columns
df[ohe_cols] = df[ohe_cols].astype(int)

print("One-Hot Encoding aplicado e convertido para inteiros.")
print("Shape atual:", df.shape)

One-Hot Encoding aplicado e convertido para inteiros.
Shape atual: (111849, 45)


## 13. Manter LIGHTCOND como ordinal inteiro

`LIGHTCOND` já vem no dataset codificada como inteiro ordinal (`0, 1, 2, 3`), representando níveis de condição de iluminação. Não há necessidade de transformação — apenas garantimos o tipo inteiro e confirmamos a distribuição.

In [13]:
df['LIGHTCOND'] = df['LIGHTCOND'].astype(int)

print("LIGHTCOND mantida como ordinal inteiro.")
display(df['LIGHTCOND'].value_counts().sort_index())

LIGHTCOND mantida como ordinal inteiro.


LIGHTCOND
0     1506
1    29514
2     5160
3    75669
Name: count, dtype: int64

## 14. Split 80/20 estratificado por SEVERITYCODE (antes do scaler)

In [14]:
X = df.drop('SEVERITYCODE', axis=1)
y = df['SEVERITYCODE']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Divisão treino/teste concluída.")
print("Proporção das classes no Treino:\n", y_train.value_counts(normalize=True).round(4).to_string())
print("\nProporção das classes no Teste:\n", y_test.value_counts(normalize=True).round(4).to_string())

Divisão treino/teste concluída.
Proporção das classes no Treino:
 SEVERITYCODE
0    0.6433
1    0.3379
2    0.0171
3    0.0017

Proporção das classes no Teste:
 SEVERITYCODE
0    0.6433
1    0.3379
2    0.0171
3    0.0017


## 15. StandardScaler nas features contínuas

Padronizamos todas as colunas **não-binárias** (mais de 2 valores distintos) — contínuas e ordinais. As colunas binárias (booleanas e dummies do One-Hot, que já são 0/1) não precisam de escala.

O `fit` é feito **apenas no treino** e o `transform` é aplicado a treino e teste, evitando data leakage (o teste não influencia a média/desvio usados na padronização).

In [15]:
cont_cols = [c for c in X_train.columns if X_train[c].nunique() > 2]

scaler = StandardScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[cont_cols] = scaler.fit_transform(X_train[cont_cols])
X_test[cont_cols] = scaler.transform(X_test[cont_cols])

print("Escalonamento concluído.")
print("Colunas escaladas:", cont_cols)
display(X_train[cont_cols].agg(['mean', 'std']).round(4))

Escalonamento concluído.
Colunas escaladas: ['longitude', 'latitude', 'PERSONCOUNT', 'PEDCOUNT', 'PEDCYLCOUNT', 'VEHCOUNT', 'LIGHTCOND', 'AWND', 'PRCP', 'SNOW', 'SNWD', 'TAVG', 'mes', 'dia_da_semana', 'hora']


,longitude,latitude,PERSONCOUNT,PEDCOUNT,PEDCYLCOUNT,VEHCOUNT,LIGHTCOND,AWND,PRCP,SNOW,SNWD,TAVG,mes,dia_da_semana,hora
mean,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 16. Verificar shape final de X_train, X_test, y_train, y_test

In [16]:
print("Resumo das dimensões (shapes):")
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

Resumo das dimensões (shapes):
X_train: (89479, 44)
X_test:  (22370, 44)
y_train: (89479,)
y_test:  (22370,)


## 17. Salvar conjuntos de treino/teste

Salvamos em `.pkl` (joblib), que preserva os tipos de dados e o índice — o Bloco 3 (modelagem) carrega esses arquivos diretamente.

In [17]:
joblib.dump(X_train, 'X_train.pkl')
joblib.dump(X_test, 'X_test.pkl')
joblib.dump(y_train, 'y_train.pkl')
joblib.dump(y_test, 'y_test.pkl')

print("Arquivos salvos com sucesso (.pkl). Pré-processamento concluído.")

Arquivos salvos com sucesso (.pkl). Pré-processamento concluído.
